# TSE 2026 — Auditoria municipal × sexo × faixa etária (v1.0)

Este notebook **não gera nova população** e **não executa a PNAD nem o R**. Ele produz uma tabela agregada com `UF × município × sexo cadastral × faixa etária`, a partir do ZIP original do TSE. O leitor trabalha em blocos para reduzir o consumo de memória.

**Antes de começar:** baixe o arquivo `simulador_eleitoral_v1_0_pacote.zip` que acompanha a versão v1.0. No Colab, mantenha a sessão anterior aberta se o arquivo original do TSE ainda estiver presente. Caso não esteja, será necessário baixá-lo novamente (aproximadamente 408 MB compactados), mas **somente esse arquivo**, não todas as bases PNAD.

A extração exige comparação exata com os 5.571 municípios e 157.826.587 eleitores domésticos já auditados. Não infere distribuição municipal de renda ou escolaridade.

## 1. Carregar o pacote de programas (pequeno)

Envie apenas `simulador_eleitoral_v1_0_pacote.zip`. Não envie aqui os 408 MB do TSE: podemos tentar localizar o download já existente na sessão.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil

base = Path('/content/simulador_eleitoral_v1_0')
base.mkdir(parents=True, exist_ok=True)

carregados = files.upload()
assert 'simulador_eleitoral_v1_0_pacote.zip' in carregados, 'Selecione o pacote v1.0'
with zipfile.ZipFile('simulador_eleitoral_v1_0_pacote.zip') as z:
    z.extractall(base)
assert (base/'codigo/extrair_conjunta_municipal_tse_v10.py').is_file()
assert (base/'inputs/tse_municipios_2026.csv').is_file()
print('Pacote instalado em:', base)

## 2. Procurar o ZIP do TSE previamente baixado

O código procura o arquivo em `/content`. Se não encontrá-lo, nenhuma extração será iniciada e nenhum dado anterior será apagado.

In [ ]:
from pathlib import Path

encontrados = [f for f in Path('/content').rglob('perfil_eleitorado_2026.zip') if f.is_file()]
print('Arquivos TSE encontrados:', len(encontrados))
for f in encontrados[:10]:
    print(f, f'{f.stat().st_size / (1024**2):.1f} MB')

TSE_ZIP = encontrados[0] if encontrados else None
print('TSE_ZIP =', TSE_ZIP)

## 3. Download **somente se necessário**

Se o ZIP original não estiver na sessão, coloque `BAIXAR_SE_FALTAR=True` e execute esta célula. O arquivo ficará guardado em `/content/simulador_eleitoral_v1_0/dados_brutos/`. Não reinicie a sessão antes de executar a extração.

In [ ]:
import subprocess

BAIXAR_SE_FALTAR = False
URL = ('https://cdn.tse.jus.br/estatistica/sead/odsele/'
       'perfil_eleitorado/perfil_eleitorado_2026.zip')
if TSE_ZIP is None:
    if not BAIXAR_SE_FALTAR:
        print('ZIP ausente. Ative BAIXAR_SE_FALTAR=True para baixar apenas o arquivo TSE.')
    else:
        destino = base/'dados_brutos/perfil_eleitorado_2026.zip'
        destino.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['curl','-fL','--retry','3','-C','-',URL,'-o',str(destino)],check=True)
        TSE_ZIP = destino
        print('Download concluído:',TSE_ZIP, destino.stat().st_size)
else:
    print('Download reutilizado:',TSE_ZIP)

## 4. Extração em blocos e auditoria de totais

Executar esta célula **apenas quando `TSE_ZIP` estiver definido**. Ela verifica o SHA-256 do arquivo original e só aceita o resultado se todos os totais por município coincidirem com o extrato oficial previamente auditado. Pode demorar alguns minutos, mas não carrega os 11 milhões de registros simultaneamente.

In [ ]:
assert TSE_ZIP is not None and TSE_ZIP.is_file(), 'Primeiro localize ou baixe o ZIP do TSE.'
import subprocess, sys

saida = base/'extracao_oficial'
subprocess.run([
    sys.executable, '-u', str(base/'codigo/extrair_conjunta_municipal_tse_v10.py'),
    '--zip', str(TSE_ZIP),
    '--tabela-municipal', str(base/'inputs/tse_municipios_2026.csv'),
    '--saida', str(saida),
    '--verificar-hash'
],check=True)

## 5. Conferência rápida e download dos resultados

In [ ]:
import pandas as pd, json, zipfile
from google.colab import files

f=saida/'tse_2026_conjunta_municipio_sexo_idade.csv.gz'
assert f.exists() and f.stat().st_size>0
dados=pd.read_csv(f)
meta=json.loads((saida/'manifesto_conjunta_municipal_v1_0.json').read_text())
print('Linhas agregadas:',len(dados),'Municípios:',meta['n_municipios'],
      'Eleitores BR:',meta['total_domestico'])
display(dados.head(12))

resultado=base/'resultado_auditoria_municipal_tse_v1_0.zip'
with zipfile.ZipFile(resultado,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for arq in sorted(saida.iterdir()):
        if arq.is_file(): z.write(arq,arcname=arq.name)
print('Resultado:',resultado)
files.download(str(resultado))